# Program 1: Pre-trained word vectors and vector arithmetic

**Aim.** Explore word relationships in pre-trained GloVe vectors by vector arithmetic, and analyse the results.

**Theory.** A word embedding places each word at a point in a vector space so that words used in similar contexts sit close together. GloVe is trained on global word co-occurrence counts. Differences between vectors can encode relations, so `king - man + woman` lands near `queen`. Closeness is measured by cosine similarity: cos(a, b) = a.b / (|a||b|).

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import numpy as np
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-100')   # 400,000 words, 100 dimensions
print(len(wv), 'words,', wv.vector_size, 'dimensions')
print('vector for "medicine" (first 8 of 100):', np.round(wv['medicine'][:8], 3))

400000 words, 100 dimensions
vector for "medicine" (first 8 of 100): [ 0.334  0.331 -0.134  0.147  0.318 -0.025  0.885 -0.126]


In [3]:
# cosine similarity by hand equals the library's value
a, b = wv['doctor'], wv['nurse']
manual = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print('doctor~nurse  manual', round(manual, 4), ' library', round(float(wv.similarity('doctor', 'nurse')), 4))
print('doctor~banana', round(float(wv.similarity('doctor', 'banana')), 4))

doctor~nurse  manual 0.7522  library 0.7522
doctor~banana 0.0117


In [4]:
# vector arithmetic: positive terms are added, negative terms subtracted, then the nearest words are listed
tests = [
  (['king', 'woman'], ['man']),
  (['paris', 'india'], ['france']),
  (['doctor', 'woman'], ['man']),
  (['walking', 'swam'], ['walked']),
  (['hospital', 'school'], ['doctor']),
]
for pos, neg in tests:
    top = wv.most_similar(positive=pos, negative=neg, topn=3)
    print(f"{' + '.join(pos)} - {' - '.join(neg)}  ->  " + ', '.join(f'{w} ({s:.2f})' for w, s in top))

king + woman - man  ->  queen (0.77), monarch (0.68), throne (0.68)
paris + india - france  ->  delhi (0.87), mumbai (0.77), bombay (0.72)
doctor + woman - man  ->  nurse (0.77), physician (0.72), doctors (0.68)
walking + swam - walked  ->  swimming (0.71), swim (0.70), biking (0.65)
hospital + school - doctor  ->  campus (0.76), schools (0.73), elementary (0.70)


In [5]:
print('odd one out:', wv.doesnt_match(['paracetamol', 'ibuprofen', 'aspirin', 'banana']))
print('odd one out:', wv.doesnt_match(['fever', 'cough', 'headache', 'hospital']))

odd one out: banana
odd one out: hospital


## Observations

Analogies work when a relation is common in the training text (capital of, gender). They are weaker for rare relations, and the arithmetic returns the nearest word, which is sometimes one of the input words. Embeddings also carry the biases of their text (see `doctor + woman - man`).